# Document Question Answering System (RAG) — Demo

This notebook walks through every stage of the RAG pipeline on the included sample document:

1. Document Ingestion
2. Text Chunking
3. Embedding Creation
4. Vector Database
5. Query Processing
6. Context Retrieval
7. Answer Generation

Assignment brief: https://docs.google.com/document/d/1TF5-Ws6QrM0C3f2l04LDCJxuml1O9y3bNR9KRjz_cLs/edit?usp=sharing

Reference implementation: https://github.com/VivekChauhan05/RAG_Document_Question_Answering

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from rag.document_loader import load_documents
from rag.chunker import chunk_documents
from rag.embeddings import get_embedder
from rag.vector_store import VectorStore
from rag.generator import generate_answer
from rag.pipeline import RAGPipeline

## Stage 1: Document Ingestion

In [ ]:
docs = load_documents(['../sample_data/sample_notes.txt'])
print(f'Loaded {len(docs)} document(s)')
print(docs[0].text[:400], '...')

## Stage 2: Text Chunking

In [ ]:
chunks = chunk_documents(docs, chunk_size=500, chunk_overlap=100)
print(f'Created {len(chunks)} chunks')
print('\n--- Example chunk ---\n')
print(chunks[0].text)

## Stage 3 & 4: Embedding Creation + Vector Database

In [ ]:
embedder = get_embedder('tfidf')  # switch to 'sentence-transformers' if installed
texts = [c.text for c in chunks]
embedder.fit(texts)
vectors = embedder.encode(texts)
print('Embedding matrix shape:', vectors.shape)

store = VectorStore()
store.add(chunks, vectors)
print(f'Vector store now holds {len(store)} chunks')

## Stage 5 & 6: Query Processing + Context Retrieval

In [ ]:
question = 'What are the main stages of a RAG pipeline?'
query_vector = embedder.encode([question])[0]
results = store.search(query_vector, top_k=3)

for chunk, score in results:
    print(f'score={score:.3f}  source={chunk.source}')
    print(chunk.text[:200], '...\n')

## Stage 7: Answer Generation

In [ ]:
context_chunks = [chunk for chunk, score in results]
answer = generate_answer(question, context_chunks, backend='extractive')
print('Q:', question)
print('A:', answer)

## All-in-one: using the `RAGPipeline` class

In [ ]:
pipeline = RAGPipeline(embedding_backend='tfidf', generation_backend='extractive', top_k=3)
pipeline.ingest(['../sample_data/sample_notes.txt'])

for q in [
    'What is RAG?',
    'Why does RAG reduce hallucination?',
    'What is hybrid search?',
]:
    result = pipeline.ask(q)
    print('Q:', q)
    print('A:', result['answer'])
    print('-' * 60)

## Next steps

- Swap `embedding_backend='tfidf'` for `'sentence-transformers'` (after `pip install sentence-transformers`) for higher quality semantic retrieval.
- Swap `generation_backend='extractive'` for `'anthropic'` or `'openai'` (with the matching API key set as an environment variable) for fluent, LLM-written answers.
- Try this pipeline on your own PDFs, or on documents derived from the Hugging Face `vectara/open_ragbench` dataset: https://huggingface.co/datasets/vectara/open_ragbench